# Values, Types & Expressions



## `val` versus `var`

Two ways to name something in Scala. They look almost identical and behave very differently.

- **`val`** — a single-assignment binding. Once given a value, the name cannot be re-pointed at a different value. This is the Scala default; reach for it first.
- **`var`** — a mutable variable. The name can be reassigned to a different value over its lifetime. Use it deliberately when you actually need a counter, an accumulator, or a state machine — and not as a reflex from another language.

The bias is real. Idiomatic Scala uses `val` for the overwhelming majority of bindings. New code that reaches for `var` should make you ask: is there a way to express this as a transformation that produces a new value, rather than a mutation that updates an old one?

In [ ]:
// val — fixed binding
val pi = 3.14159
// pi = 3.14  // would not compile: "reassignment to val"

// var — mutable binding
var counter = 0
counter = counter + 1
counter = counter + 1
println(counter)  // 2

// You can reassign a var to a different value of a compatible type, but
// not to a value of an incompatible type — the type was inferred at the
// first assignment and is fixed.
var n = 10
n = 99       // ok, both Int
// n = "ten" // would not compile: type mismatch

Two subtleties worth pinning down now, because both come back later.

**`val` locks the binding, not the value it points at.** If you bind a `val` to a mutable object — say, a `scala.collection.mutable.ArrayBuffer`, or a java `ArrayList`, or a Scala `Array` — the binding is fixed but the contents of the object can still change in place. We will see this concretely when we meet `Array` in notebook 04. The takeaway: immutability is a property of the *type*, not of the keyword `val`.

**The right-hand side is evaluated immediately.** When you write `val x = expensive()`, `expensive()` runs at the point of the `val` declaration, not the first time you read `x`. If you want lazy evaluation, that's `lazy val`, covered next.

## `lazy val` — computed at most once, on first read

A `lazy val` is a binding whose right-hand side is not evaluated when the declaration is reached. It runs the first time the name is actually used, and the result is then cached for every subsequent read.

Two common reasons to reach for it:

- The computation is expensive and might not be needed at all in some code paths.
- The computation depends on something that isn't ready yet at declaration time but will be later.

Behind the scenes the compiler generates a private field for the cached result and a thread-safe `synchronized` initialization block. So `lazy val` is safe to share across threads — at the cost of a small first-access overhead.

In [ ]:
var sideEffectsObserved = 0

val eager: Int =
  sideEffectsObserved += 1
  println("eager evaluated")
  42

lazy val lazyOne: Int =
  sideEffectsObserved += 1
  println("lazy evaluated")
  99

println(s"after declarations, side effects = $sideEffectsObserved")  // 1 — only `eager` ran

println(s"first read of lazyOne: ${lazyOne}")   // prints "lazy evaluated", then 99
println(s"second read of lazyOne: ${lazyOne}")  // no print — cached
println(s"final side effects = $sideEffectsObserved")  // 2

## The type hierarchy

Every value in Scala is an instance of a type, and every type sits somewhere in a single rooted hierarchy. Knowing the shape of that hierarchy explains a lot of subsequent behavior — what the compiler accepts, what `null` does, what an empty list's element type is, why `throw` can appear in a position that expects a value.

```text
                          Any
                          / \
                     AnyVal   AnyRef        <- AnyRef == java.lang.Object
                    /  |  |     |  \
                 Int  Boolean   String  ... every other class
                 Long Char            (user-defined classes too)
                 Float Unit            \
                 ...   ...              Null         <- subtype of every AnyRef
                                          \
                                        Nothing      <- subtype of EVERY type
```

The hierarchy in words.

- **`Any`** is the root. Every value is an `Any`. It defines a handful of universal methods like `==`, `!=`, `hashCode`, `toString`.
- **`AnyVal`** is the parent of the *value types* — these are the types backed by java virtual machine primitives (`Int`, `Long`, `Double`, `Boolean`, `Char`, `Byte`, `Short`, `Float`) plus the strange one-value type `Unit`.
- **`AnyRef`** is the parent of every *reference type* — every class, including the standard library's `String`, `List`, `Map`, and every class you define. `AnyRef` is exactly `java.lang.Object` under the hood.
- **`Null`** is the type of the literal `null`, and it sits as a subtype of every reference type. You can assign `null` to a `String` (don't), but you cannot assign `null` to an `Int` — `Null` is not a subtype of `AnyVal`.
- **`Nothing`** is the bottom — a subtype of *every* type, including the value types. Nothing has no values. It is the type of expressions that never return, like `throw new Exception(...)`. We will use this idea in notebook 06 when we meet `Option`.

In [ ]:
val anything: Any = 42
val anything2: Any = "forty two"
val anything3: Any = List(1, 2, 3)

// Nothing in action — `throw` is an expression of type Nothing,
// so it type-checks in any position that expects any type at all.
def crashAndBurn(msg: String): Int =
  throw new RuntimeException(msg)
  // The compiler is happy: throw has type Nothing, which is a subtype of Int.

// Null in action — assignable to AnyRef subtypes, but flagged by `-Yexplicit-nulls`
// in modern Scala 3 setups. Idiomatic Scala uses Option (notebook 06) instead.
val maybeName: String = null
println(maybeName)  // prints "null" — and any method call on it will NPE

## Type inference — what the compiler does for free

Scala is statically typed, but the compiler does most of the typing work for you. For a `val`, the compiler reads the right-hand side, computes its type, and uses that as the type of the binding. No annotation needed.

```scala
val a = 42              // inferred Int
val b = 3.14            // inferred Double
val c = "scala"         // inferred String
val d = List(1, 2, 3)   // inferred List[Int]
val e = (1, "one", 1.0) // inferred (Int, String, Double)
```

Most of the code you read and write will look like that — no explicit types in sight. But there are three places where you *should* write the type explicitly even when inference would succeed.

## Three places to write the type explicitly

**On public method signatures.** A method's parameter and return types are part of its contract. Hide the types and a downstream change to the body silently changes the type the world depends on. Always annotate the return type of a `def` that other code calls.

**When inference picks something wider than you intended.** A common one: `val xs = List(1, 2.0, "three")` infers `List[Matchable]` (or similar wide upper bound) — almost never what you wanted. Writing `val xs: List[Any] = ...` makes the intent explicit, or restructure to avoid the heterogeneous list.

**When the right-hand side is recursive or mutually recursive.** Recursive `def`s need a return type annotation to break the inference cycle. Without it, the compiler errors with "recursive method needs result type."

The honest rule: let inference handle local `val`s in method bodies, but annotate every public-facing signature.

In [ ]:
// 1. Public method — explicit return type
def normaliseEmail(raw: String): String =
  raw.trim.toLowerCase

// 2. Recursive method — must annotate the return type
def factorial(n: Long): Long =
  if n <= 1 then 1
  else n * factorial(n - 1)

// 3. Forced widening — make intent explicit
val numbers: List[Number] = List(1, 2.0, 3L)  // explicit upper bound

// Local val in a method body — let inference do its work
def stats(xs: List[Int]): (Int, Double) =
  val total = xs.sum                      // inferred Int
  val mean  = total.toDouble / xs.length  // inferred Double
  (total, mean)

## The numeric types

Scala's numeric types are thin wrappers over java virtual machine primitives. Their sizes and ranges are exactly what the java virtual machine provides.

| Type      | Size    | Range                                              | Literal |
|-----------|---------|----------------------------------------------------|---------|
| `Byte`    | 8 bit   | -128 to 127                                        | `42.toByte` |
| `Short`   | 16 bit  | -32 768 to 32 767                                  | `42.toShort` |
| `Int`     | 32 bit  | roughly ±2.1 billion                               | `42` (default integer literal) |
| `Long`    | 64 bit  | roughly ±9.2 quintillion                           | `42L` |
| `Float`   | 32 bit  | IEEE 754 single precision                          | `3.14f` |
| `Double`  | 64 bit  | IEEE 754 double precision                          | `3.14` (default decimal literal) |
| `BigInt`  | arbitrary | unbounded integer, slower, on the heap          | `BigInt("12345...")` |
| `BigDecimal` | arbitrary | unbounded decimal, exact decimal arithmetic | `BigDecimal("0.1")` |

Three things you have to remember when working with these.

**Default literal types.** An integer literal like `42` is an `Int`, not a `Long`. A decimal literal like `3.14` is a `Double`, not a `Float`. Add the `L` or `f` suffix to override.

**Overflow is silent.** Adding past the maximum of an `Int` wraps around to a large negative number. The compiler will not warn you. Reach for `Long` when the magnitudes can grow, or `BigInt` when they can grow without bound.

**Floating point is not decimal.** `0.1 + 0.2` produces `0.30000000000000004` in `Double`, exactly as it does in every other IEEE-754 language. For money or any context where exact decimal arithmetic matters, reach for `BigDecimal`.

In [ ]:
// Silent overflow
val almostMax: Int = Int.MaxValue
println(almostMax)            //  2147483647
println(almostMax + 1)        // -2147483648  — wrapped, no error

// Promotion is one-directional: smaller widens to larger automatically,
// the other direction requires an explicit conversion.
val smallInt: Int  = 100
val asLong: Long   = smallInt        // ok, Int widens to Long
val asDouble: Double = smallInt      // ok, Int widens to Double
// val backToInt: Int = asLong       // would not compile — explicit needed:
val backToInt: Int = asLong.toInt    // ok

// Floating point reality
println(0.1 + 0.2)            // 0.30000000000000004

// BigDecimal — exact decimal arithmetic
val a = BigDecimal("0.1")
val b = BigDecimal("0.2")
println(a + b)                // 0.3 — exact

## `String` and the built-in interpolators

A `String` in Scala is a `java.lang.String` — exact same class, exact same methods. There is no separate Scala string type. What Scala adds is a small set of *string interpolators* — prefixes you put before the opening quote that change how the literal is parsed.

The three built-in ones:

- **`s"..."`** — splice expressions with `$name` and `${expr}`.
- **`f"..."`** — splice with `printf`-style format specifiers attached.
- **`raw"..."`** — splice like `s`, but don't process backslash escapes (so `\n` stays as two characters).

There is also the triple-quote form `"""..."""` for multi-line strings that contain quotes, backslashes, and newlines without any escaping. It composes with the interpolator prefixes — `s"""..."""` works.

In [ ]:
val name = "ganesh"
val score = 92.345

// s — plain interpolation
println(s"hello, $name")
println(s"length = ${name.length}")          // wrap arbitrary expressions in ${...}

// f — printf-style formatting
println(f"score: $score%.2f")                // score: 92.35  (rounded to 2 dp)
println(f"name: $name%-10s | score: $score%6.2f")  // padded fields

// raw — no escape processing
println(raw"line1\nline2")                  // prints the four characters: \nline2
println(s"line1\nline2")                    // prints two lines

// triple-quoted multi-line
val sql =
  s"""SELECT id, amount
     |FROM transactions
     |WHERE customer = '$name'""".stripMargin
println(sql)

The `stripMargin` method in the last example is a stylistic helper. Triple-quoted strings preserve indentation; `stripMargin` removes leading whitespace up to and including the `|` character on each line. The default margin character is `|`; you can pass a different one if you have a string that contains pipes.

**What interpolators compile to.** An interpolator is a method call on the `StringContext` class. When you write `s"hello $name"`, the compiler rewrites it to `StringContext("hello ", "").s(name)`. This means interpolators are not magic — you can write your own by extending `StringContext`. Notebook 08 (extensions) revisits this.

## `Boolean`, `Char`, `Unit`, `Null`, `Nothing`

The non-numeric value types and the strange residents at the edges of the hierarchy.

- **`Boolean`** — `true` and `false`. Same as java's `boolean`, backed by a one-byte primitive.
- **`Char`** — a 16-bit unsigned integer interpreted as a UTF-16 code unit. Written `'a'`, `'\n'`, `'\u00e9'`. Because it is 16-bit, characters outside the Basic Multilingual Plane (most emoji, some CJK extensions) require *two* `Char`s to represent — a surrogate pair. Practical consequence: `"😀".length == 2`. For real Unicode handling, work with `String` and use code-point methods.
- **`Unit`** — the type with exactly one value, written `()`. Used as the return type of methods that exist for their side effects. Equivalent to java's `void`, except `Unit` is a real type you can write down.
- **`Null`** — the type of the literal `null`. Subtype of every reference type. Modern Scala 3 setups can opt into *explicit nulls* with the `-Yexplicit-nulls` compiler flag, which makes `null` no longer assignable to a `String` (etc.) without an explicit nullable type. We will recommend `Option` (notebook 06) instead in every situation where you would otherwise reach for `null`.
- **`Nothing`** — the bottom type. No values. The inferred type of `throw ...`, of `???` (the standard placeholder for "not implemented yet"), and of an `Option` of nothing in particular (`Option.empty[Nothing]`). Anywhere an expression of type `Nothing` is used, the compiler accepts it as any type, because `Nothing` is a subtype of everything.

In [ ]:
// Boolean
val flag: Boolean = true
println(!flag && false)   // false

// Char — careful with surrogate pairs
val letter: Char = 'a'
println("😀".length)       // 2 — one emoji, two UTF-16 code units

// Unit — the type of side-effecting methods
def log(msg: String): Unit =
  println(s"[log] $msg")
val nothingValue: Unit = ()  // the sole value

// Nothing — used by ??? and throw
def notDoneYet(): String = ???  // throws NotImplementedError when called

// `???` is just a method that throws; its return type is Nothing,
// so it type-checks anywhere — handy for stubbing out methods while
// you write the rest of the program.

## Type aliases

A `type` declaration gives a name to an existing type. It does **not** create a new type — the alias and the original are interchangeable everywhere.

Two reasons to use them:

- **Readability.** A repeated complex type — say, `Map[String, List[(Int, Double)]]` — becomes one named thing.
- **Encoding intent.** A `UserId` is conceptually different from a plain `String`, even when its representation is identical. The alias signals that.

When you need *real* nominal separation — where the compiler refuses to let you pass a plain `String` where a `UserId` is expected — reach for an opaque type or a value class. Notebook 07 covers both.

In [ ]:
type UserId      = String
type CustomerMap = Map[UserId, List[String]]

def lookup(map: CustomerMap, id: UserId): List[String] =
  map.getOrElse(id, Nil)

val data: CustomerMap = Map(
  "u1" -> List("invoice-001", "invoice-002"),
  "u2" -> List("invoice-003"),
)

println(lookup(data, "u1"))  // List(invoice-001, invoice-002)

// Aliases are transparent — a UserId IS a String
val raw: String = "u1"
val id: UserId  = raw   // no conversion needed
println(id == raw)      // true

## The big shift — almost everything is an expression

In imperative languages like Python or java, `if` is a *statement*: it controls flow but does not have a value. To get a value out, you assign inside both branches.

In Scala, `if` is an **expression**: it evaluates to a value, and you bind that value with a `val`. The same is true for `for ... yield`, for block expressions, and for pattern matching.

This is a small mental shift with large downstream consequences for how Scala code reads. You stop writing "do this, then assign that" sequences. You start writing "the value of this is the result of choosing between these alternatives."

The one big exception is `while` — it is a statement-only loop that always evaluates to `Unit`. You will see why in a moment.

In [ ]:
// if as an expression — both branches contribute a value
val score = 73
val grade: String =
  if score >= 90 then "A"
  else if score >= 75 then "B"
  else if score >= 60 then "C"
  else "F"
println(grade)  // C

// for-yield as an expression — produces a collection
val squares: List[Int] =
  for n <- List(1, 2, 3, 4, 5) yield n * n
println(squares)  // List(1, 4, 9, 16, 25)

// for-yield with a filter
val evenSquares: List[Int] =
  for
    n <- List(1, 2, 3, 4, 5)
    if n % 2 == 0
  yield n * n
println(evenSquares)  // List(4, 16)

// A block — the value of the block is the value of its last expression
val area: Double =
  val r      = 3.0
  val pi     = 3.14159
  pi * r * r       // <- value of the block
println(area)  // 28.27431

## `while` — the one statement-only construct

`while` (and its sibling `do-while`) is the exception that proves the rule: it always evaluates to `Unit`. There is nothing useful to "return" from a loop whose only purpose is to iterate by side effect.

Idiomatic Scala uses `while` sparingly — mostly in tight numeric loops where the alternative (a `foreach` or recursive call) would have measurable overhead. For everything else, reach for `for-yield`, `map`/`filter`/`fold`, or recursion. Notebook 04 covers the higher-level alternatives.

In [ ]:
// A while loop — Unit value, side effects only
var i = 0
val loopResult: Unit =
  while i < 3 do
    println(s"iteration $i")
    i += 1
println(s"loopResult = $loopResult")  // loopResult = ()

// The same thing, expression-style, without var or while
(0 until 3).foreach(j => println(s"iteration $j"))
// or, if you wanted a value out:
val collected: IndexedSeq[String] = (0 until 3).map(j => s"iteration $j")
println(collected)

## Operators are method calls

There are no operator declarations in Scala. What looks like an operator — `+`, `-`, `*`, `<`, `==`, even `::` and `++` — is a method call written in a special syntactic position.

The rule: if a method takes one argument, you can call it without the dot and parentheses. So `1 + 2` is exactly `1.+(2)`. And `list1 ++ list2` is exactly `list1.++(list2)`.

This is why Scala lets you define methods with symbolic names. A class can declare a `def +(other: Money): Money` and use it like a built-in operator. Notebook 05 will use this to add operators to domain types.

In [ ]:
// Operators in infix position vs method position — identical
println(1 + 2)             // 3
println(1.+(2))            // 3

println("hello " + "world")
println("hello ".+("world"))

// Comparison operators are also methods
println(3.<(5))            // true
println(List(1).==(List(1)))  // true — structural equality on case-class-like types

// You can define your own
final class Money(val cents: Long):
  def +(other: Money): Money = Money(cents + other.cents)
  override def toString: String = f"$$${cents / 100}%d.${cents % 100}%02d"

val a = Money(500)   // 5 dollars
val b = Money(125)   // 1 dollar 25 cents
println(a + b)       // $6.25
println((a + b).cents)  // 625

## Type ascription — `expr: Type`

A type ascription tells the compiler "evaluate this expression as this type." It does not change the runtime value; it changes how the compiler treats it for subsequent inference.

Three honest uses:

- **Forcing widening.** `val xs = List(1, 2, 3)` is `List[Int]`. `val xs = List(1, 2, 3): List[Any]` is `List[Any]`. The runtime value is the same.
- **Disambiguating overloaded methods.** When two overloads differ only by argument type, ascribing the argument picks one.
- **Signalling intent on a return.** Inside a long block, an ascription on the final expression documents the type the block is expected to produce, and the compiler enforces it.

In [ ]:
val ints: List[Int]  = List(1, 2, 3)
val anys: List[Any]  = List(1, 2, 3): List[Any]

println(ints)  // List(1, 2, 3)
println(anys)  // List(1, 2, 3)

// Type ascription is not a cast. It only succeeds when the expression
// is already a subtype of the ascribed type.
val s: String = "hello"
val a: Any = s          // implicit widening, always ok
val b: Any = s: Any     // explicit ascription, ok
// val c: Int = s: Int  // would not compile — String is not an Int